# Cómo convertir un agente en una API, con FastAPI

## ¿Por qué una API?
| Sin API | Con API |
|---|---|
| El agente solo corre en el notebook | Cualquier app (web, móvil, bot) puede usarlo |
| Solo lo usa quien tiene el código | Se expone como servicio HTTP estándar |
| Sin control de acceso | Cada cliente tiene su propia API Key |


## Arquitectura de la API

```
Cliente (curl / requests / app)
        │
        │  POST /query  {"query": "..."}
        │  Header: X-API-Key: agent_xxxxx
        ▼
┌─────────────────────────┐
│      FastAPI (main.py)  │
│  ┌───────────────────┐  │
│  │  Middleware Auth  │  │  ← verifica la API Key
│  └────────┬──────────┘  │
│           ▼             │
│  ┌───────────────────┐  │
│  │  LlamaIndex Agent │  │  ← elige la tool correcta
│  └────────┬──────────┘  │
│           ▼             │
│  ┌───────────────────┐  │
│  │  Tools / APIs     │  │  ← ejecuta la acción
│  └───────────────────┘  │
└─────────────────────────┘
        │
        │  {"response": "...", "trace": [...]}
        ▼
     Cliente
```

## Estructura del proyecto

- main.py: donde está el agente
- requirements.txt: archivo que indica que librerías de python está usando mi progama
- .env: las claves secretas
- sesion_14.ipynb: la explicación

## 1. Cómo generar una API Key Segura

In [1]:
import secrets

def generate_api_key(prefix: str = "agente_") -> str:
    """Generates a secure API key with a specified prefix."""
    return prefix + secrets.token_urlsafe(64)
 

In [2]:
api_key_generada = generate_api_key()
print("API Key generada:")
print(api_key_generada)

API Key generada:
agente_euewoGS1KIzGcXn16XZtGjiiQWUULgOEO152BY4FGPY_7NMURtl_VfEXncs6JZs0ntjdkaK5pBawTPGBNa1c7g


## 2. Crear archivo .env

Se ve de la siguiente manera:

```ini
# .env
OPENAI_API_KEY=sk-proj-...
AGENT_API_KEYS=agent_clave1,agent_clave2
```

## 3. Crear archivo main.py
El código está en la carpeta de este notebook

## 4. Probar la API desde Python

In [3]:
import requests
from IPython.display import display, HTML

In [ ]:
BASE_URL = "https://agente-llama-index.onrender.com/"

# API KEY PARA AUTENTICACIÓN
API_KEY = ""

headers = {"X-API-Key": API_KEY}    

In [5]:
def mostrar_respuesta(query: str, data: dict) -> None:
    """Muestra la respuesta del agente de forma visual."""
    html = ['<div style="font-family:sans-serif;border:1px solid #ccc;padding:12px;margin:8px 0;border-radius:6px">']
    html.append(f'<p><strong>Consulta:</strong> {query}</p>')
    html.append(f'<p><strong>Respuesta:</strong> {data.get("response", "")}</p>')
    trace = data.get("trace", [])
    if trace:
        html.append('<p><strong>Trace de tools:</strong></p><ol>')
        for paso in trace:
            html.append(
                f'<li><code>{paso["tool"]}</code>({paso["args"]}) '
                f'&rarr; <em>{paso["output"]}</em></li>'
            )
        html.append("</ol>")
    else:
        html.append("<p><em>El agente respondió sin invocar tools.</em></p>")
    html.append("</div>")
    display(HTML("".join(html)))

In [6]:
query = "¿Cuál es el clima en Madrid hoy?"
response = requests.post(
    f"{BASE_URL}/query",
    json={"query": query},
    headers=headers
)

print ("Respuesta del agente:")
if response.status_code == 200:
    mostrar_respuesta(query, response.json())

Respuesta del agente:


In [7]:
query = "¿Cuál es el clima en la capital de Noruega el día de hoy?"
response = requests.post(
    f"{BASE_URL}/query",
    json={"query": query},
    headers=headers
)

print ("Respuesta del agente:")
if response.status_code == 200:
    mostrar_respuesta(query, response.json())

Respuesta del agente:


In [8]:
query = "¿Cuántos dólares son 100 euros hoy?"
response = requests.post(
    f"{BASE_URL}/query",
    json={"query": query},
    headers=headers
)

print ("Respuesta del agente:")
if response.status_code == 200:
    mostrar_respuesta(query, response.json())

Respuesta del agente:
